In [ ]:
!pip install unsloth -q

In [ ]:
from google.colab import files
import json
import os
import torch
import re
from typing import Dict, List, Set, Tuple, Optional
from collections import defaultdict
from datasets import Dataset
from unsloth import FastLanguageModel

In [ ]:
print("загрузка workout_lora_model.zip, test.jsonl, exercises.json")
uploaded = files.upload()
if "workout_lora_model.zip" in uploaded:
    !unzip -o workout_lora_model.zip -d ./

def load_catalog(filepath: str) -> Dict[str, Dict]:
    with open(filepath, 'r', encoding='utf-8') as f:
        exercises_list = json.load(f)
    return {ex["id"]: ex for ex in exercises_list}

CATALOG_BY_ID = load_catalog("exercises.json")

model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing=False,
)
model.load_adapter("workout_lora_model", adapter_name="default")
model.set_adapter("default")

In [ ]:
SYSTEM_PROMPT = """You are a fitness AI trainer. Create safe workouts in JSON format.

CRITICAL RULES:
1. Return ONLY valid JSON, no other text, no markdown
2. Use ONLY exercise_id from catalog
3. Every exercise MUST include ALL required fields

STRENGTH needs: exercise_id, exercise_type="strength", sets(1-10), reps(1-50), weight_kg
CARDIO needs: exercise_id, exercise_type="cardio", duration_minutes(1-120), pace(walk/jog/run/sprint/recovery)
YOGA needs: exercise_id, exercise_type="yoga", hold_seconds(5-300), breath_count(1-20)

Example: {"workout_name": "Leg Day", "type": "strength", "duration_min": 30, "exercises": [{"exercise_id": "squat", "exercise_type": "strength", "sets": 3, "reps": 10, "weight_kg": null}]}"""


In [ ]:
def test_model(prompt: str):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.2,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response:
        response = response.split("assistant")[-1].strip()
    return response

def load_jsonl(file_path: str) -> List[Dict]:
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

In [ ]:
def extract_duration_from_prompt(prompt: str) -> Optional[int]:
    patterns = [
        r'(\d+)\s*мин', r'(\d+)\s*минут', r'полчаса', r'час', r'полтора часа',
    ]
    for pattern in patterns:
        match = re.search(pattern, prompt.lower())
        if match:
            if 'полчаса' in match.group(): return 30
            elif 'час' in match.group() and 'полтора' in prompt.lower(): return 90
            elif 'час' in match.group(): return 60
            return int(match.group(1))
    return None

def extract_constraints_from_prompt(prompt: str) -> Dict:
    constraints = {"level": None, "contraindications": [], "location": None}
    if any(w in prompt.lower() for w in ["новичок", "начинаю", "базовый", "с нуля"]):
        constraints["level"] = "beginner"
    elif any(w in prompt.lower() for w in ["продвинутый", "спортсмен", "опытный"]):
        constraints["level"] = "advanced"
    elif any(w in prompt.lower() for w in ["средний", "регулярно"]):
        constraints["level"] = "intermediate"

    injury_map = {
        "колен": "knee_injury", "колено": "knee_injury",
        "плеч": "shoulder_injury", "плечо": "shoulder_injury",
        "спин": "lower_back_pain", "поясниц": "lower_back_pain",
        "запяст": "wrist_injury", "ше": "neck_injury",
        "давлен": "high_blood_pressure", "беремен": "pregnancy"
    }
    for kw, injury in injury_map.items():
        if kw in prompt.lower():
            constraints["contraindications"].append(injury)

    if "дома" in prompt.lower() or "домашн" in prompt.lower():
        constraints["location"] = "home"
    elif "зал" in prompt.lower() or "тренаж" in prompt.lower():
        constraints["location"] = "gym"
    return constraints

def get_semantic_group(exercise_id: str, catalog_by_id: Dict) -> Tuple[str, str]:
    ex = catalog_by_id.get(exercise_id, {})
    muscle = ex.get("muscleGroup", "unknown")
    eid = exercise_id.lower()
    if any(w in eid for w in ["squat", "присед", "goblet"]): pattern = "squat_pattern"
    elif any(w in eid for w in ["lunge", "выпад", "step-up"]): pattern = "lunge_pattern"
    elif any(w in eid for w in ["press", "жим", "bench", "push"]): pattern = "push_pattern"
    elif any(w in eid for w in ["pull", "тяг", "row", "chin-up"]): pattern = "pull_pattern"
    elif any(w in eid for w in ["plank", "планк", "crunch", "скруч"]): pattern = "core_stability"
    elif any(w in eid for w in ["yoga", "поз", "асан", "stretch"]): pattern = "flexibility"
    elif any(w in eid for w in ["walk", "run", "jog", "бег", "ходьб"]): pattern = "locomotion"
    else: pattern = "other"
    return muscle, pattern

def calculate_semantic_similarity(expected_ids: Set[str], predicted_ids: Set[str], catalog_by_id: Dict) -> float:
    if not expected_ids and not predicted_ids: return 1.0
    if not expected_ids or not predicted_ids: return 0.0
    expected_groups = {get_semantic_group(eid, catalog_by_id) for eid in expected_ids}
    predicted_groups = {get_semantic_group(eid, catalog_by_id) for eid in predicted_ids}
    intersection = len(expected_groups & predicted_groups)
    union = len(expected_groups | predicted_groups)
    return intersection / union if union > 0 else 0.0

def check_safety_compliance(predicted_exercises: List[Dict], user_constraints: Dict, catalog_by_id: Dict) -> Tuple[bool, List[str]]:
    violations = []
    user_contra = set(user_constraints.get("contraindications", []))
    for ex in predicted_exercises:
        eid = ex.get("exercise_id")
        cat = catalog_by_id.get(eid, {})
        ex_contra = set(cat.get("contraindications", []))
        if user_contra & ex_contra:
            violations.append(f"{eid} противопоказан при {user_contra & ex_contra}")
    return len(violations) == 0, violations

def validate_exercise_fields(exercise: Dict, workout_type: str) -> Tuple[bool, List[str]]:
    errors = []
    required = ["exercise_id", "exercise_type"]
    if workout_type == "strength":
        required += ["sets", "reps"]
        for field in required:
            if field not in exercise or exercise[field] is None:
                errors.append(f"Missing: {field}")
        if isinstance(exercise.get("sets"), int) and not (1 <= exercise["sets"] <= 10):
            errors.append(f"sets out of range")
        if isinstance(exercise.get("reps"), int) and not (1 <= exercise["reps"] <= 50):
            errors.append(f"reps out of range")
    elif workout_type == "cardio":
        required += ["duration_minutes", "pace"]
        for field in required:
            if field not in exercise or exercise[field] is None:
                errors.append(f"Missing: {field}")
        if exercise.get("pace") not in ["walk", "jog", "run", "sprint", "recovery"]:
            errors.append(f"Invalid pace")
    elif workout_type == "yoga":
        required += ["hold_seconds", "breath_count"]
        for field in required:
            if field not in exercise or exercise[field] is None:
                errors.append(f"Missing: {field}")
    return len(errors) == 0, errors

In [ ]:

def evaluate_on_test(test_data: List[Dict], model, tokenizer, catalog_by_id: Dict, threshold_bad: float = 0.5) -> Dict:
    metrics = {
        "total": len(test_data), "valid_json": 0, "correct_type": 0, "correct_duration": 0, "safe_predictions": 0, "valid_fields": 0, "jaccard_semantic": [], "bad_examples": []
    }

    for idx, ex in enumerate(test_data):
        user_msg = ex["messages"][1]["content"]
        expected = json.loads(ex["messages"][2]["content"])
        expected_ids = {e["exercise_id"] for e in expected.get("exercises", [])}
        user_constraints = extract_constraints_from_prompt(user_msg)

        response = test_model(user_msg)

        try:
            pred = json.loads(response)
            metrics["valid_json"] += 1

            if pred.get("type") == expected.get("type"):
                metrics["correct_type"] += 1

            req_dur = extract_duration_from_prompt(user_msg)
            pred_dur = pred.get("duration_min", 0)
            if req_dur and abs(pred_dur - req_dur) <= 5:
                metrics["correct_duration"] += 1

            all_fields_valid = True
            pred_exercises = pred.get("exercises", [])
            for ex_pred in pred_exercises:
                is_valid, _ = validate_exercise_fields(ex_pred, pred.get("type"))
                if not is_valid:
                    all_fields_valid = False
                    break
            if all_fields_valid and pred_exercises:
                metrics["valid_fields"] += 1

            is_safe, violations = check_safety_compliance(pred_exercises, user_constraints, catalog_by_id)
            if is_safe:
                metrics["safe_predictions"] += 1
            else:
                metrics["bad_examples"].append({
                    "index": idx, "user_msg": user_msg[:100],
                    "issue": "SAFETY_VIOLATION", "violations": violations,
                    "predicted_ids": [e["exercise_id"] for e in pred_exercises]
                })
                continue

            pred_ids = {e["exercise_id"] for e in pred_exercises}

            jaccard_semantic = calculate_semantic_similarity(expected_ids, pred_ids, catalog_by_id)
            metrics["jaccard_semantic"].append(jaccard_semantic)

            if jaccard_semantic < threshold_bad:
                metrics["bad_examples"].append({
                    "index": idx, "user_msg": user_msg[:100],
                    "expected_ids": list(expected_ids), "predicted_ids": list(pred_ids),
                    "jaccard_semantic": round(jaccard_semantic, 3)
                })

        except json.JSONDecodeError:
            metrics["bad_examples"].append({
                "index": idx, "user_msg": user_msg[:100],
                "error": "invalid_json", "response_snippet": response[:200]
            })
        except Exception as e:
            metrics["bad_examples"].append({
                "index": idx, "user_msg": user_msg[:100],
                "error": f"runtime: {str(e)[:50]}"
            })

    n = metrics["total"]
    results = {
        "valid_json_pct": metrics["valid_json"] / n * 100 if n else 0,
        "correct_type_pct": metrics["correct_type"] / n * 100 if n else 0,
        "correct_duration_pct": metrics["correct_duration"] / n * 100 if n else 0,
        "safe_predictions_pct": metrics["safe_predictions"] / n * 100 if n else 0,
        "valid_fields_pct": metrics["valid_fields"] / n * 100 if n else 0,
        "avg_jaccard_semantic": sum(metrics["jaccard_semantic"]) / len(metrics["jaccard_semantic"]) * 100 if metrics["jaccard_semantic"] else 0,
        "bad_examples_count": len(metrics["bad_examples"]),
        "bad_examples": metrics["bad_examples"][:10]
    }
    print(f"валидный JSON: {results['valid_json_pct']:.1f}%")
    print(f"правильный тип: {results['correct_type_pct']:.1f}%")
    print(f"точная длительность: {results['correct_duration_pct']:.1f}%")
    print(f"валидные поля: {results['valid_fields_pct']:.1f}%")
    print(f"безопасные предсказания: {results['safe_predictions_pct']:.1f}%")
    print(f"Jaccard: {results['avg_jaccard_semantic']:.1f}% ← ОСНОВНАЯ")

    return results


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 111.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 115.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 123.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

Saving exercises.json to exercises.json
Saving test.jsonl to test.jsonl
Saving workout_lora_model.zip to workout_lora_model.zip
✅ Загружено: exercises.json
✅ Загружено: test.jsonl
✅ Загружено: workout_lora_model.zip
Archive:  workout_lora_model.zip
   creating: ./workout_lora_model/
  inflating: ./workout_lora_model/README.md  
  inflating: ./workout_lora_model/adapter_config.json  
  inflating: ./workout_lora_model/tokenizer.json  
  inflating: ./workout_lora_model/adapter_model.safetensors  
  inflating: ./workout_lora_model/tokenizer_config.json  
  inflating: ./workout_lora_model/chat_template.jinja  
✅ Модель распакована
✅ Каталог загружен: 30 упражнений
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!

🚀 Загрузка базовой модели + ваших LoRA-адаптеров...
==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platfo

model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

...

✅ Модель и адаптеры загружены

🔄 Загрузка тестовых данных...
✅ Загружено 395 тестовых примеров

🚀 Запуск оценки...


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12


📊 ОТЧЁТ (395 примеров)
✅ Валидный JSON:           100.0%
✅ Правильный тип:          82.3%
✅ Точная длительность:     91.4%
✅ Валидные поля:           99.7%
🛡️  Безопасные предсказания: 87.6% ⚠️
📏 Jaccard (точный):        51.9%
🧠 Jaccard (семантический): 60.1% ← ОСНОВНАЯ
❌ Плохие примеры:          170

🔍 ТОП-3 проблемных случая:

1. [1] Создай strength тренировку на спины, 15 минут, beginner уровень
   ⚠️  НАРУШЕНИЕ БЕЗОПАСНОСТИ: ["barbell-row противопоказан при {'lower_back_pain'}", "deadlift противопоказан при {'lower_back_pain'}"]

2. [3] Создай йога-комплекс! дыхание, 15 мин, опытный атлет? с ограничениями: коленные суставы
   ⚠️  НАРУШЕНИЕ БЕЗОПАСНОСТИ: ["child-pose противопоказан при {'knee_injury'}"]

3. [7] Очень нужно! Дай практику для укрепить тело! Время 30 мин, скручивания!!!
   📊 Jaccard: точный=0.0, семантический=0.0

💾 Результаты сохранены в evaluation_results.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
test_data = load_jsonl("test.jsonl")
results = evaluate_on_test(
    test_data=test_data,
    model=model,
    tokenizer=tokenizer,
    catalog_by_id=CATALOG_BY_ID,
    threshold_bad=0.4
)